# Lecture 9: State-Space Models and the Kalman Filter
In this notebook we will cover:
- State-space representation
- Kalman filter algorithm (theory + manual implementation)
- Local linear trend model
- Handling missing observations
- Kalman smoother


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve
import warnings
warnings.filterwarnings('ignore')

# filterpy is optional
try:
    from filterpy.kalman import KalmanFilter as FPKalmanFilter
    FILTERPY = True
except ImportError:
    FILTERPY = False

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print(f"filterpy: {'available' if FILTERPY else 'not installed (manual implementation will be used)'}")


## 9.1 State-Space Representation

General linear Gaussian state-space model:

**Observation equation:**
$$Y_t = G_t \mathbf{x}_t + W_t, \quad W_t \sim N(0, R_t)$$

**State equation:**
$$\mathbf{x}_t = F_t \mathbf{x}_{t-1} + V_t, \quad V_t \sim N(\mathbf{0}, Q_t)$$

**Initialisation:** $\mathbf{x}_0 \sim N(\mathbf{m}_0, P_0)$

| Symbol | Dimension | Meaning |
|--------|-----------|---------|
| $\mathbf{x}_t$ | $m \times 1$ | Unobserved state |
| $Y_t$ | $1 \times 1$ | Observation |
| $F_t$ | $m \times m$ | State transition matrix |
| $G_t$ | $1 \times m$ | Observation matrix |
| $Q_t$ | $m \times m$ | Process noise covariance |
| $R_t$ | scalar | Measurement noise variance |


In [ ]:
# ── Kalman Filter: Manual Implementation ──

def kalman_filter(Y, F, G, Q, R, x0, P0):
    # F: state transition matrix (m x m)
    # G: observation matrix (1 x m)
    # Q: process noise covariance (m x m)
    # R: measurement noise variance (scalar)
    # x0, P0: initial state and covariance
    n = len(Y)
    m = len(x0)
    
    x_pred = np.zeros((n, m))
    P_pred = np.zeros((n, m, m))
    x_filt = np.zeros((n, m))
    P_filt = np.zeros((n, m, m))
    K_all  = np.zeros((n, m))
    innov  = np.zeros(n)
    
    x_k = x0.copy()
    P_k = P0.copy()
    
    for t in range(n):
        # Predict step
        x_p = F @ x_k
        P_p = F @ P_k @ F.T + Q
        
        # Innovation
        v_t = Y[t] - G @ x_p
        S_t = G @ P_p @ G.T + R    # Innovation variance
        
        # Kalman gain
        K_t = (P_p @ G.T) / S_t
        
        # Update step
        x_k = x_p + K_t * v_t
        P_k = (np.eye(m) - np.outer(K_t, G)) @ P_p
        
        # Store
        x_pred[t] = x_p
        P_pred[t] = P_p
        x_filt[t] = x_k
        P_filt[t] = P_k
        K_all[t]  = K_t
        innov[t]  = v_t
    
    return x_filt, P_filt, x_pred, P_pred, innov, K_all

print("Kalman filter function ready.")


## 9.2 Simple Example: AR(1) Model as a State-Space System

The AR(1) model $X_t = \phi X_{t-1} + Z_t$ in state-space form:
- $F = [\phi]$, $G = [1]$, $Q = [\sigma^2_Z]$, $R = 0$

With noisy observations ($Y_t = X_t + W_t$): $R = \sigma^2_W > 0$


In [ ]:
# ── AR(1) + Noisy Observation Filtering ──

np.random.seed(42)
n = 150
phi = 0.85
sigma_z = 1.0
sigma_w = 2.0  # Observation noise (high)

# True state
X_true = np.zeros(n)
for t in range(1, n):
    X_true[t] = phi*X_true[t-1] + np.random.normal(0, sigma_z)

# Noisy observation
Y_obs = X_true + np.random.normal(0, sigma_w, n)

# Kalman filter parameters
F = np.array([[phi]])
G = np.array([1.0])
Q = np.array([[sigma_z**2]])
R = sigma_w**2
x0 = np.array([0.0])
P0 = np.array([[sigma_z**2 / (1 - phi**2)]])

x_filt, P_filt, x_pred, P_pred, innov, K = kalman_filter(Y_obs, F, G, Q, R, x0, P0)
sigma_filt = np.sqrt(P_filt[:, 0, 0])

fig, axes = plt.subplots(3, 1, figsize=(13, 10))

axes[0].plot(Y_obs, color='gray', lw=0.7, alpha=0.7, label='Noisy Observation $Y_t$')
axes[0].plot(X_true, color='steelblue', lw=1.5, label='True State $X_t$')
axes[0].plot(x_filt[:,0], color='red', lw=1.5, ls='--', label='Kalman Filter')
axes[0].fill_between(range(n),
                      x_filt[:,0]-2*sigma_filt,
                      x_filt[:,0]+2*sigma_filt,
                      alpha=0.2, color='red', label='95% CI')
axes[0].set_title('Kalman Filter: AR(1) State Recovery', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(K[:,0], color='purple', lw=1.2)
axes[1].set_title('Kalman Gain $K_t$ (Converges Over Time)', fontweight='bold')
axes[1].axhline(K[-1,0], color='red', ls='--', alpha=0.5, label=f'Steady state={K[-1,0]:.3f}')
axes[1].legend()

axes[2].plot(innov, color='darkorange', lw=0.8, alpha=0.8)
axes[2].axhline(0, color='black', ls='--')
axes[2].set_title('Innovation $v_t = Y_t - G\hat{x}_{t|t-1}$ (should be WN)', fontweight='bold')

plt.tight_layout()
plt.show()

rmse_obs = np.sqrt(np.mean((Y_obs - X_true)**2))
rmse_filt = np.sqrt(np.mean((x_filt[:,0] - X_true)**2))
print(f"Raw observation RMSE : {rmse_obs:.3f}")
print(f"Kalman filter RMSE   : {rmse_filt:.3f}")
print(f"Improvement          : {100*(rmse_obs-rmse_filt)/rmse_obs:.1f}%")


## 9.3 Local Linear Trend Model

$$Y_t = \mu_t + \epsilon_t, \quad \epsilon_t \sim N(0, \sigma^2_\epsilon)$$
$$\mu_t = \mu_{t-1} + \beta_{t-1} + \xi_t, \quad \xi_t \sim N(0, \sigma^2_\xi)$$
$$\beta_t = \beta_{t-1} + \zeta_t, \quad \zeta_t \sim N(0, \sigma^2_\zeta)$$

State vector: $\mathbf{x}_t = (\mu_t, \beta_t)^T$

$$F = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}, \quad G = \begin{pmatrix} 1 & 0 \end{pmatrix}$$


In [ ]:
# ── Local Linear Trend + Kalman ──

np.random.seed(42)
n = 200

# True trend simulation
sigma_eps  = 2.0
sigma_xi   = 0.3
sigma_zeta = 0.05

mu   = np.zeros(n)
beta = np.zeros(n)
mu[0], beta[0] = 5.0, 0.1

for t in range(1, n):
    beta[t] = beta[t-1] + np.random.normal(0, sigma_zeta)
    mu[t]   = mu[t-1] + beta[t-1] + np.random.normal(0, sigma_xi)

Y_llt = mu + np.random.normal(0, sigma_eps, n)

# State-space parameters
F_llt = np.array([[1.0, 1.0],
                   [0.0, 1.0]])
G_llt = np.array([1.0, 0.0])
Q_llt = np.diag([sigma_xi**2, sigma_zeta**2])
R_llt = sigma_eps**2
x0_llt = np.array([Y_llt[0], 0.0])
P0_llt = np.diag([10.0, 1.0])

x_filt_llt, P_filt_llt, _, _, innov_llt, _ = kalman_filter(
    Y_llt, F_llt, G_llt, Q_llt, R_llt, x0_llt, P0_llt)

sig_mu = np.sqrt(P_filt_llt[:, 0, 0])

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(Y_llt, color='gray', lw=0.7, alpha=0.6, label='Observation $Y_t$')
axes[0].plot(mu, color='steelblue', lw=2, label='True Trend $\mu_t$')
axes[0].plot(x_filt_llt[:,0], color='red', lw=1.5, ls='--', label='Kalman Filtered Trend')
axes[0].fill_between(range(n),
                      x_filt_llt[:,0]-2*sig_mu,
                      x_filt_llt[:,0]+2*sig_mu,
                      alpha=0.2, color='red')
axes[0].set_title('Local Linear Trend: Trend Extraction via Kalman Filter', fontweight='bold')
axes[0].legend()

axes[1].plot(beta, color='steelblue', lw=1.5, label='True Slope $\\beta_t$')
axes[1].plot(x_filt_llt[:,1], color='red', lw=1.5, ls='--', label='Estimated Slope')
axes[1].set_title('Slope Component $\\beta_t$', fontweight='bold')
axes[1].axhline(0, color='black', ls='--', alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


## 9.4 Handling Missing Observations

A key feature of the Kalman filter is that **the update step can be skipped for missing observations** — only the predict step is executed.

$$Y_t = \text{NaN} \Rightarrow \hat{x}_{t|t} = \hat{x}_{t|t-1}, \quad P_{t|t} = P_{t|t-1}$$


In [ ]:
# ── Kalman Filter with Missing Observations ──

def kalman_filter_missing(Y, F, G, Q, R, x0, P0):
    n = len(Y)
    m = len(x0)
    x_filt = np.zeros((n, m))
    P_filt = np.zeros((n, m, m))
    x_k, P_k = x0.copy(), P0.copy()
    
    for t in range(n):
        # Predict step
        x_p = F @ x_k
        P_p = F @ P_k @ F.T + Q
        
        if np.isnan(Y[t]):
            # Skip update; carry forward the prediction
            x_k = x_p
            P_k = P_p
        else:
            v_t = Y[t] - G @ x_p
            S_t = G @ P_p @ G.T + R
            K_t = (P_p @ G.T) / S_t
            x_k = x_p + K_t * v_t
            P_k = (np.eye(m) - np.outer(K_t, G)) @ P_p
        
        x_filt[t] = x_k
        P_filt[t] = P_k
    
    return x_filt, P_filt

# Data: set 30% of observations to missing
Y_missing = Y_llt.copy()
missing_idx = np.random.choice(n, size=int(0.30*n), replace=False)
Y_missing[missing_idx] = np.nan

x_filt_miss, P_filt_miss = kalman_filter_missing(
    Y_missing, F_llt, G_llt, Q_llt, R_llt, x0_llt, P0_llt)

sig_miss = np.sqrt(P_filt_miss[:, 0, 0])

fig, ax = plt.subplots(figsize=(13, 5))
ax.scatter(range(n), Y_llt, color='gray', s=8, alpha=0.5, label='Complete Data')
ax.scatter(range(n), Y_missing, color='steelblue', s=10, alpha=0.8, label='Observed')
ax.plot(x_filt_miss[:,0], color='red', lw=1.5, label='Kalman (Missing)', zorder=5)
ax.fill_between(range(n),
                x_filt_miss[:,0]-2*sig_miss,
                x_filt_miss[:,0]+2*sig_miss,
                alpha=0.2, color='red')
ax.set_title('Kalman Filter with Missing Observations (30% Missing)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total observations   : {n}")
print(f"Missing observations : {len(missing_idx)} ({100*len(missing_idx)/n:.0f}%)")
print(f"Complete data RMSE   : {np.sqrt(np.mean((x_filt_llt[:,0]-mu)**2)):.3f}")
print(f"Missing data RMSE    : {np.sqrt(np.mean((x_filt_miss[:,0]-mu)**2)):.3f}")


## Summary

| Step | Operation |
|------|-----------|
| **Predict** | $\hat{x}_{t\|t-1} = F\hat{x}_{t-1}$, $P_{t\|t-1} = FP_{t-1}F^T + Q$ |
| **Innovation** | $v_t = Y_t - G\hat{x}_{t\|t-1}$ |
| **Kalman Gain** | $K_t = P_{t\|t-1}G^T / (GP_{t\|t-1}G^T + R)$ |
| **Update** | $\hat{x}_t = \hat{x}_{t\|t-1} + K_t v_t$ |
| **Covariance Update** | $P_t = (I - K_tG)P_{t\|t-1}$ |

**Applications:**
- GPS positioning
- Financial trend extraction
- Robot odometry
- Economic indicator estimation (with missing data)
